# Pipeline de avaliação — ferramenta

> **Comentário de organização:** este notebook é um **exemplo pronto para rodar** no layout organizado da pasta `eval/`. Ele consolida o caminho esperado para novas execuções:
>
> - notebook em `eval/notebooks/`
> - perguntas em `eval/perguntas/`
> - resultados em `eval/resultados_ferramenta/`
> - agente em `agent/`, na raiz do projeto
>
> As execuções reais já realizadas foram arquivadas em `eval/backup_run/`. Esses arquivos de backup vêm de uma organização anterior e podem ter **paths diferentes**; ajuste os caminhos antes de tentar reexecutá-los.

Este notebook roda o agente para combinações de edital × modelo, salva um `.xlsx` por combinação em `eval/resultados_ferramenta/` e gera um sumário consolidado.

Por segurança, `SOBRESCREVER = False` por padrão: se um resultado final já existir, a combinação é pulada. Para reexecutar e substituir arquivos existentes, altere explicitamente para `True` na célula de configuração.

In [ ]:
import os, sys, time, glob, re
from pathlib import Path
from datetime import datetime

import warnings
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
from dotenv import load_dotenv
from langchain_core.callbacks import BaseCallbackHandler


def encontrar_roots(start=None) -> tuple[Path, Path]:
    """
    Encontra:
      - PROJECT_ROOT: raiz do repositório, onde fica `agent/`
      - EVAL_ROOT: pasta da avaliação, onde ficam `perguntas/` e `resultados_ferramenta/`

    Layout esperado:
      edital-assistant/
        agent/
        eval/
          notebooks/
          perguntas/
          resultados_ferramenta/
          backup_run/
    """
    start = Path(start or Path.cwd()).resolve()

    tentativas: list[tuple[Path, Path]] = []
    for p in [start, *start.parents]:
        # Caso comum: p é a raiz do projeto.
        tentativas.append((p, p / "eval"))

        # Caso o notebook seja aberto com cwd dentro de eval/.
        tentativas.append((p.parent, p))

        # Fallback para layouts antigos.
        tentativas.append((p, p / "evals"))
        tentativas.append((p, p))

    vistos = set()
    for project_root, eval_root in tentativas:
        project_root = project_root.resolve()
        eval_root = eval_root.resolve()
        chave = (project_root, eval_root)
        if chave in vistos:
            continue
        vistos.add(chave)

        if (project_root / "agent").is_dir() and (eval_root / "perguntas").is_dir():
            return project_root, eval_root

    raise RuntimeError(
        "Não encontrei a estrutura esperada. Ajuste manualmente, por exemplo:\n"
        "PROJECT_ROOT = Path('/home/julio/Documentos/tcc_GENAI/v8/edital-assistant')\n"
        "EVAL_ROOT = PROJECT_ROOT / 'eval'"
    )


PROJECT_ROOT, EVAL_ROOT = encontrar_roots()
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from agent.agent import build_agent, ask

print(f"cwd:          {Path.cwd()}")
print(f"project_root: {PROJECT_ROOT}")
print(f"eval_root:    {EVAL_ROOT}")
print(f"env:          {PROJECT_ROOT / '.env'}")

## TokenTracker — captura tokens + cache

In [ ]:
class TokenTracker(BaseCallbackHandler):
    def __init__(self):
        self.reset()

    def reset(self):
        self.input_tokens = 0
        self.output_tokens = 0
        self.cache_read_tokens = 0
        self.cache_write_tokens = 0
        self.n_calls = 0

    def on_llm_end(self, response, **kwargs):
        self.n_calls += 1

        # 1) Formato OpenAI/DeepSeek via llm_output.token_usage
        usage = (getattr(response, "llm_output", None) or {}).get("token_usage") or {}
        if usage:
            self.input_tokens += usage.get("prompt_tokens", 0) or 0
            self.output_tokens += usage.get("completion_tokens", 0) or 0

            ds_hit = usage.get("prompt_cache_hit_tokens") or 0
            details = usage.get("prompt_tokens_details") or {}
            oai_hit = details.get("cached_tokens") or 0
            self.cache_read_tokens += max(ds_hit, oai_hit)
            return

        # 2) Formato LangChain novo via message.usage_metadata
        for gen_list in response.generations:
            for gen in gen_list:
                msg = getattr(gen, "message", None)
                um = getattr(msg, "usage_metadata", None) if msg else None
                if um:
                    self.input_tokens += um.get("input_tokens", 0) or 0
                    self.output_tokens += um.get("output_tokens", 0) or 0

                    details = um.get("input_token_details") or {}
                    self.cache_read_tokens += details.get("cache_read", 0) or 0
                    self.cache_write_tokens += details.get("cache_creation", 0) or 0
                    break

## Configuração

A lista abaixo representa a execução completa, mas você pode comentar modelos ou editais para rodar em lotes menores.

- `MAX_PERGUNTAS = None`: usa todas as perguntas de cada arquivo.
- `SOBRESCREVER = False`: não substitui resultados já existentes.
- `RODAR_SMOKE_TEST = True`: testa uma pergunta simples por combinação antes da execução principal.

In [ ]:
MODELOS = [
    # (provider, modelo)
    ("openai",    "gpt-4o-mini"),
    ("deepseek",  "deepseek-v4-flash"),
    ("openai",    "gpt-5.4-mini"),
    ("deepseek",  "deepseek-v4-pro"),
    ("anthropic", "claude-haiku-4-5"),
    ("openai",    "gpt-5.4"),
    ("anthropic", "claude-sonnet-4-6"),
    ("anthropic", "claude-opus-4-7"),
    ("openai",    "gpt-5.5"),
]

EDITAIS = [
    ("bndes",     1),
    ("cvm",       2),
    ("petrobras", 3),
]

# USD por 1M tokens. Ordem: (input_miss, input_cached, output).
PRECOS = {
    "openai/gpt-4o-mini":             (0.15,  0.075,  0.60),
    "openai/gpt-5.4-mini":            (0.25,  0.125,  2.00),
    "openai/gpt-5.4":                 (1.25,  0.625, 10.00),
    "openai/gpt-5.5":                 (5.00,  2.500, 30.00),
    "deepseek/deepseek-v4-flash":     (0.14,  0.028,  0.28),
    "deepseek/deepseek-v4-pro":       (1.74,  0.145,  3.48),
    "anthropic/claude-haiku-4-5":     (1.00,  1.000,  5.00),
    "anthropic/claude-sonnet-4-6":    (3.00,  3.000, 15.00),
    "anthropic/claude-opus-4-7":      (5.00,  5.000, 25.00),
}

PERGUNTAS_DIR = EVAL_ROOT / "perguntas"
RESULTADOS_DIR = EVAL_ROOT / "resultados_ferramenta"
RESULTADOS_DIR.mkdir(parents=True, exist_ok=True)

MAX_PERGUNTAS = None           # use 1 para teste rápido; None roda todas
LIMITE_USD_POR_COMBO = 20.00   # teto pessimista por combinação, sem considerar cache
SOBRESCREVER = False           # deixe False para não substituir resultados existentes
RODAR_SMOKE_TEST = True        # recomendado antes da execução principal

print(f"Perguntas:            {PERGUNTAS_DIR}")
print(f"Resultados:           {RESULTADOS_DIR}")
print(f"Modelos ativos:       {len(MODELOS)}")
print(f"Editais ativos:       {len(EDITAIS)}")
print(f"Total combinações:    {len(MODELOS) * len(EDITAIS)}")
print(f"Max perguntas/combo:  {MAX_PERGUNTAS}")
print(f"Sobrescrever:         {SOBRESCREVER}")
print(f"Smoke test:           {RODAR_SMOKE_TEST}")

## Validações gratuitas

Esta célula não chama APIs. Ela verifica preços, arquivos de perguntas e possíveis arquivos de saída já existentes.

In [ ]:
# 1) Todos os modelos têm preço configurado.
faltando_precos = [f"{p}/{m}" for p, m in MODELOS if f"{p}/{m}" not in PRECOS]
if faltando_precos:
    raise RuntimeError(f"Modelos sem preço em PRECOS: {faltando_precos}")

# 2) Arquivos de perguntas existem.
faltando_perguntas = [
    str(PERGUNTAS_DIR / f"{edital_nome}.xlsx")
    for edital_nome, _ in EDITAIS
    if not (PERGUNTAS_DIR / f"{edital_nome}.xlsx").exists()
]
if faltando_perguntas:
    raise RuntimeError("Arquivos de perguntas não encontrados:\n- " + "\n- ".join(faltando_perguntas))

print("OK — preços e arquivos de perguntas encontrados.\n")
print("Arquivos de saída já existentes:")

for edital_nome, _ in EDITAIS:
    encontrados = sorted(RESULTADOS_DIR.glob(f"{edital_nome}_*.xlsx"))
    print(f"\n{edital_nome}: {len(encontrados)} arquivo(s)")
    for arq in encontrados[:30]:
        print(f"  - {arq.name}")
    if len(encontrados) > 30:
        print(f"  ... +{len(encontrados) - 30} arquivo(s)")

## Helpers

In [ ]:
def slug(modelo: str) -> str:
    return modelo.replace("/", "_").replace(":", "_")


def custo_estimado(in_tok: int, out_tok: int, p_in: float, p_out: float) -> float:
    """Teto pessimista: trata todo input como cache miss."""
    return in_tok / 1_000_000 * p_in + out_tok / 1_000_000 * p_out


def custo_real(in_tok: int, out_tok: int, cache_read: int,
               p_in: float, p_in_cached: float, p_out: float) -> float:
    miss = max(in_tok - cache_read, 0)
    return (
        miss / 1_000_000 * p_in
        + cache_read / 1_000_000 * p_in_cached
        + out_tok / 1_000_000 * p_out
    )


def caminho_saida(provider: str, modelo: str, edital_nome: str, parcial: bool = False) -> Path:
    """
    Padrão final usado em eval/resultados_ferramenta:
      bndes_gpt-4o-mini.xlsx
      cvm_claude-opus-4-7.xlsx
      petrobras_deepseek-v4-flash.xlsx
    """
    sufixo = "__PARCIAL" if parcial else ""
    return RESULTADOS_DIR / f"{edital_nome}_{slug(modelo)}{sufixo}.xlsx"


def limpar_saida_existente(provider: str, modelo: str, edital_nome: str) -> int:
    """Remove saída final/parcial dessa combinação somente quando SOBRESCREVER=True."""
    removidos = 0
    for parcial in [False, True]:
        arq = caminho_saida(provider, modelo, edital_nome, parcial=parcial)
        if arq.exists():
            arq.unlink()
            removidos += 1
    return removidos


def salvar_xlsx(provider: str, modelo: str, edital_nome: str,
                resultados: list[dict], parcial: bool = False) -> Path:
    arq = caminho_saida(provider, modelo, edital_nome, parcial=parcial)

    if arq.exists() and not SOBRESCREVER:
        raise FileExistsError(
            f"Arquivo já existe e SOBRESCREVER=False: {arq}\n"
            "Altere SOBRESCREVER=True ou remova/renomeie o arquivo antes de reexecutar."
        )

    df_r = pd.DataFrame(resultados)
    df_r.to_excel(arq, index=False)
    return arq

In [ ]:
def carregar_perguntas(edital_nome: str) -> pd.DataFrame:
    arq_perguntas = PERGUNTAS_DIR / f"{edital_nome}.xlsx"
    df_q = pd.read_excel(arq_perguntas)

    if "pergunta" not in df_q.columns:
        raise RuntimeError(f"{arq_perguntas} não tem coluna 'pergunta'.")

    df_q = df_q.dropna(subset=["pergunta"])
    df_q = df_q[df_q["pergunta"].astype(str).str.strip() != ""].reset_index(drop=True)

    if MAX_PERGUNTAS is not None:
        df_q = df_q.head(MAX_PERGUNTAS).copy()

    if "id" not in df_q.columns:
        df_q.insert(0, "id", range(1, len(df_q) + 1))

    return df_q


def rodar_combo(provider: str, modelo: str, edital_nome: str, edital_id: int,
                limite_usd: float = LIMITE_USD_POR_COMBO):
    """
    Roda as perguntas configuradas. Retorna (xlsx_path, erro_fatal_ou_None).

    Erros em perguntas individuais vão para a coluna `erro` e a combinação continua.
    Estouro de orçamento interrompe a combinação e salva parcial.
    """
    chave = f"{provider}/{modelo}"
    p_in, p_in_cached, p_out = PRECOS[chave]

    df_q = carregar_perguntas(edital_nome)
    print(f"Perguntas carregadas para {edital_nome}: {len(df_q)}")

    agente = build_agent(provider=provider, model=modelo)
    tracker = TokenTracker()
    agente.llm = agente.llm.with_config(callbacks=[tracker])
    agente.llm_check = agente.llm_check.with_config(callbacks=[tracker])

    resultados = []
    custo_estim_acum = 0.0
    erro_fatal = None

    for i, row in df_q.iterrows():
        pergunta = row["pergunta"]
        t0 = time.time()
        erro, resposta = None, None

        tracker.reset()
        try:
            resposta = ask(
                agent=agente,
                question=pergunta,
                chat_history=[],
                edital_id_ativo=edital_id,
            )
        except Exception as e:
            erro = str(e)

        in_tok = tracker.input_tokens
        out_tok = tracker.output_tokens
        cache_read = tracker.cache_read_tokens
        n_calls = tracker.n_calls
        latencia = round(time.time() - t0, 2)

        c_estim = custo_estimado(in_tok, out_tok, p_in, p_out)
        c_real = custo_real(in_tok, out_tok, cache_read, p_in, p_in_cached, p_out)
        custo_estim_acum += c_estim

        resultados.append({
            "id": row["id"],
            "categoria": row.get("categoria", ""),
            "pergunta": pergunta,
            "resposta": resposta,
            "input_tokens": in_tok,
            "output_tokens": out_tok,
            "total_tokens": in_tok + out_tok,
            "cache_read_tokens": cache_read,
            "cache_miss_tokens": max(in_tok - cache_read, 0),
            "n_invocacoes": n_calls,
            "custo_usd": round(c_estim, 6),
            "custo_real_usd": round(c_real, 6),
            "latencia_s": latencia,
            "erro": erro,
        })

        flag = "ERRO" if erro else "ok"
        cache_pct = (cache_read / in_tok * 100) if in_tok else 0
        print(
            f"      [{i+1:>2}/{len(df_q)}] {flag:>4}  {latencia:>5}s  "
            f"calls={n_calls}  in={in_tok:>5} (cache {cache_pct:>4.0f}%)  "
            f"out={out_tok:>4}  acum=${custo_estim_acum:.4f}"
        )

        if custo_estim_acum > limite_usd:
            erro_fatal = f"Estouro na pergunta {i+1}: ${custo_estim_acum:.4f} > ${limite_usd}"
            break

    arq = salvar_xlsx(provider, modelo, edital_nome, resultados, parcial=bool(erro_fatal)) if resultados else None
    return arq, erro_fatal

## Smoke test

Esta célula chama a API uma vez por combinação ativa. Se alguma combinação falhar por credencial, nome de modelo ou tool calling, a execução principal não começa.

In [ ]:
if RODAR_SMOKE_TEST:
    print("=== SMOKE TEST ===\n")

    smoke_falhas = []

    for provider, modelo in MODELOS:
        for edital_nome, edital_id in EDITAIS:
            print(f"  [test] {provider}/{modelo} × {edital_nome} ... ", end="", flush=True)
            try:
                agente = build_agent(provider=provider, model=modelo)
                tracker = TokenTracker()
                agente.llm = agente.llm.with_config(callbacks=[tracker])
                agente.llm_check = agente.llm_check.with_config(callbacks=[tracker])

                resp = ask(
                    agent=agente,
                    question="Qual o salário inicial?",
                    chat_history=[],
                    edital_id_ativo=edital_id,
                )

                if not resp or len(resp.strip()) < 20:
                    raise ValueError(f"resposta vazia/curta: {resp!r}")

                print(f"ok ({tracker.input_tokens} in / {tracker.output_tokens} out / {tracker.n_calls} calls)")
            except Exception as e:
                smoke_falhas.append(f"{provider}/{modelo} × {edital_nome}: {e}")
                print(f"FALHOU\n           {e}")

    if smoke_falhas:
        raise RuntimeError(
            f"\n{len(smoke_falhas)} combinação(ões) falhou(ram). Corrija ou comente:\n  - "
            + "\n  - ".join(smoke_falhas)
        )

    print(f"\nTodas as {len(MODELOS) * len(EDITAIS)} combinações passaram.")
else:
    print("RODAR_SMOKE_TEST=False — smoke test pulado.")

## Pipeline principal

Esta célula chama a API para todas as perguntas das combinações ativas.

Com `SOBRESCREVER=False`, combinações que já têm arquivo final em `resultados_ferramenta/` são puladas. Isso permite deixar o notebook no repositório sem risco de apagar resultados já produzidos.

In [ ]:
sucessos, parciais, falhas, pulados = [], [], [], []

for provider, modelo in MODELOS:
    for edital_nome, edital_id in EDITAIS:
        arq_final = caminho_saida(provider, modelo, edital_nome, parcial=False)
        arq_parcial = caminho_saida(provider, modelo, edital_nome, parcial=True)

        if not SOBRESCREVER and arq_final.exists():
            pulados.append((provider, modelo, edital_nome, arq_final))
            print(f"\n[skip] {provider}/{modelo} × {edital_nome} — já existe: {arq_final.name}")
            continue

        if SOBRESCREVER:
            n = limpar_saida_existente(provider, modelo, edital_nome)
            if n:
                print(f"\n[del]  {n} arquivo(s) antigo(s) de {provider}/{modelo} × {edital_nome}")
        elif arq_parcial.exists():
            pulados.append((provider, modelo, edital_nome, arq_parcial))
            print(f"\n[skip] {provider}/{modelo} × {edital_nome} — parcial já existe: {arq_parcial.name}")
            continue

        print(f"\n[run]  {provider}/{modelo} × {edital_nome} (id={edital_id})")

        try:
            arq, erro_fatal = rodar_combo(provider, modelo, edital_nome, edital_id)
        except Exception as e:
            falhas.append((provider, modelo, edital_nome, str(e)))
            print(f"       ✗ FALHOU antes do xlsx: {e}")
            continue

        if erro_fatal:
            parciais.append((provider, modelo, edital_nome, arq, erro_fatal))
            print(f"       ⚠ PARCIAL — {erro_fatal}\n         salvo em {arq.name}")
        else:
            sucessos.append((provider, modelo, edital_nome, arq))
            print(f"       ✓ ok — salvo em {arq.name}")

print("\n" + "=" * 60)
print("=== SUMÁRIO DA EXECUÇÃO ===")
print("=" * 60)

print(f"\n✓ Sucessos: {len(sucessos)}")
for p, m, e, a in sucessos:
    print(f"    {p}/{m} × {e}  →  {a.name}")

print(f"\n⚠ Parciais: {len(parciais)}")
for p, m, e, a, err in parciais:
    print(f"    {p}/{m} × {e}  →  {a.name}\n        {err}")

print(f"\n↷ Pulados: {len(pulados)}")
for p, m, e, a in pulados:
    print(f"    {p}/{m} × {e}  →  {a.name}")

print(f"\n✗ Falhas: {len(falhas)}")
for p, m, e, err in falhas:
    print(f"    {p}/{m} × {e}\n        {err}")

## Sumário consolidado dos resultados

Lê os `.xlsx` em `eval/resultados_ferramenta/` e monta uma tabela com custo, latência e erros por arquivo.

In [ ]:
linhas = []

for arq in sorted(RESULTADOS_DIR.glob("*.xlsx")):
    # Ignora arquivos temporários do Excel, se existirem.
    if arq.name.startswith("~$"):
        continue

    try:
        df = pd.read_excel(arq)
    except Exception as e:
        print(f"Ignorando {arq.name}: erro ao ler ({e})")
        continue

    if df.empty:
        continue

    if "cache_read_tokens" not in df.columns:
        df["cache_read_tokens"] = 0
    if "custo_real_usd" not in df.columns:
        df["custo_real_usd"] = df.get("custo_usd", 0)
    if "erro" not in df.columns:
        df["erro"] = None

    nome_sem_ext = arq.stem.replace("__PARCIAL", "")
    partes = nome_sem_ext.split("_", 1)
    edital = partes[0] if partes else ""
    modelo = partes[1] if len(partes) > 1 else ""

    input_total = int(df["input_tokens"].sum()) if "input_tokens" in df.columns else 0
    cache_total = int(df["cache_read_tokens"].sum()) if "cache_read_tokens" in df.columns else 0

    linhas.append({
        "edital": edital,
        "modelo": modelo,
        "n_perguntas": len(df),
        "n_erros": int(df["erro"].notna().sum()),
        "input_tokens": input_total,
        "output_tokens": int(df["output_tokens"].sum()) if "output_tokens" in df.columns else 0,
        "cache_read": cache_total,
        "cache_pct": cache_total / max(input_total, 1),
        "custo_estim_usd": round(float(df["custo_usd"].sum()), 6) if "custo_usd" in df.columns else 0,
        "custo_real_usd": round(float(df["custo_real_usd"].sum()), 6),
        "latencia_med_s": round(float(df["latencia_s"].mean()), 2) if "latencia_s" in df.columns else None,
        "latencia_p95_s": round(float(df["latencia_s"].quantile(0.95)), 2) if "latencia_s" in df.columns else None,
        "parcial": "__PARCIAL" in arq.stem,
        "arquivo": arq.name,
    })

df_sumario = pd.DataFrame(linhas)

if not df_sumario.empty:
    df_sumario = df_sumario.sort_values(["edital", "modelo"]).reset_index(drop=True)
    df_sumario["cache_pct"] = df_sumario["cache_pct"].map("{:.1%}".format)

df_sumario

## Tabela cruzada — custo real por modelo × edital

In [ ]:
if df_sumario.empty:
    print("Nenhum resultado encontrado para montar a tabela cruzada.")
else:
    df_pivot = (
        df_sumario
        .pivot_table(index="modelo", columns="edital", values="custo_real_usd", aggfunc="sum")
        .fillna(0)
    )
    df_pivot["total"] = df_pivot.sum(axis=1)
    df_pivot = df_pivot.sort_values("total")
    display(df_pivot)
    print(f"Total geral estimado nos arquivos lidos: ${df_pivot['total'].sum():.6f}")